# Семинар 6. LangChain / LangGraph — code-first notebook

Этот ноутбук по-прежнему устроен как последовательность запускаемых ячеек сверху вниз, но теперь между кодом есть короткие пояснения.  
Фокус остаётся тем же: `ChatOpenAI`-совместимая модель, `ChatPromptTemplate`, structured output, `@tool`, `create_agent`, `checkpointer`, `StateGraph`, `InMemoryStore`.

Важно держать в голове простую лестницу абстракций:

1. **messages / model** — самый низкий уровень работы с чатом  
2. **prompt + parser + LCEL chain** — уже удобнее собирать повторяемые пайплайны  
3. **structured output** — модель начинает возвращать не просто текст, а объект фиксированной формы  
4. **tools + agent** — модель получает право вызывать внешние функции  
5. **memory + graph** — появляется состояние, продолжение диалога и явная оркестрация шагов

Примеры здесь завязаны не на заглушки, а на реальные HTTP API:

- weather: **Open-Meteo**
- calculator: **math.js web API**

Лучший режим запуска — просто идти по ячейкам по порядку и смотреть на объекты, которые печатаются после каждого шага.

In [1]:
!pip -q install -U langchain langgraph langchain-openai requests pydantic grandalf


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:

from __future__ import annotations

import inspect
import json
import os
import uuid
from dataclasses import dataclass
from pprint import pprint
from typing import Any

import requests
from pydantic import BaseModel, Field

from langchain.agents import AgentState, create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.store.memory import InMemoryStore

SESSION = requests.Session()


def jprint(obj: Any) -> None:
    if isinstance(obj, BaseModel):
        obj = obj.model_dump()
    print(json.dumps(obj, ensure_ascii=False, indent=2))


def show_message(msg: Any) -> None:
    if hasattr(msg, "pretty_repr"):
        print(msg.pretty_repr())
    else:
        print(msg)


def show_last_message(result: dict[str, Any]) -> None:
    messages = result["messages"]
    show_message(messages[-1])


### Импорты и карта модулей

Здесь сразу видно, как разделяются роли библиотек:

- `langchain_*` — высокоуровневые абстракции: модель, prompts, tools, agents;
- `langgraph.*` — состояние, checkpointing, граф и store;
- `pydantic` — схема входа и выхода;
- `requests` — реальный I/O наружу.

Полезно замечать этот разрез с самого начала.  
Когда код начинает расти, становится проще понимать, в каком слое появилась проблема: в модели, в tool, в state, в graph или в обычной HTTP-функции.

In [4]:

# Модель можно подключить как к OpenAI, так и к любому OpenAI-compatible endpoint.
# Для seminar setup удобно использовать env-переменные.
#
# Примеры:
# export OPENAI_API_KEY="..."
# export OPENAI_BASE_URL="https://api.openai.com/v1"
#
# или:
# export LITELLM_API_KEY="..."
# export LITELLM_BASE_URL="http://host:port/v1"

MODEL_NAME = 
API_KEY = 
BASE_URL =

if not API_KEY:
    raise ValueError(
        "Нужен OPENAI_API_KEY или LITELLM_API_KEY в переменных окружения."
    )

model_kwargs = {
    "model": MODEL_NAME,
    "api_key": API_KEY,
    "temperature": 0,
}
if BASE_URL:
    model_kwargs["base_url"] = BASE_URL

model = ChatOpenAI(**model_kwargs)

print("ChatOpenAI class:")
print(ChatOpenAI)
print("\nChatOpenAI signature:")
print(inspect.signature(ChatOpenAI))
print("\nModel object:")
print(model)


ChatOpenAI class:
<class 'langchain_openai.chat_models.base.ChatOpenAI'>

ChatOpenAI signature:
(*args: Any, name: str | None = None, cache: langchain_core.caches.BaseCache | bool | None = None, verbose: bool = <factory>, callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None, tags: list[str] | None = None, metadata: dict[str, typing.Any] | None = None, custom_get_token_ids: collections.abc.Callable[[str], list[int]] | None = None, rate_limiter: langchain_core.rate_limiters.BaseRateLimiter | None = None, disable_streaming: Union[bool, Literal['tool_calling']] = False, output_version: str | None = <factory>, profile: langchain_core.language_models.model_profile.ModelProfile | None = None, client: Any = None, async_client: Any = None, root_client: Any = None, root_async_client: Any = None, model: str = 'gpt-3.5-turbo', temperature: float | None = None, model_kwargs: dict[str, typing.Any] = <factory>, api_key: p

In [5]:

print("HumanMessage class:", HumanMessage)
print("AIMessage class:", AIMessage)
print("SystemMessage class:", SystemMessage)

messages = [
    SystemMessage(content="Отвечай кратко и технически."),
    HumanMessage(content="Что такое LangChain в одном абзаце?"),
]

response = model.invoke(messages)

print("\nResponse object type:", type(response))
print("\nResponse object:")
show_message(response)


HumanMessage class: <class 'langchain_core.messages.human.HumanMessage'>
AIMessage class: <class 'langchain_core.messages.ai.AIMessage'>
SystemMessage class: <class 'langchain_core.messages.system.SystemMessage'>

Response object type: <class 'langchain_core.messages.ai.AIMessage'>

Response object:
================================== Ai Message ==================================

LangChain — это фреймворк для построения приложений на основе языковых моделей, позволяющий интегрировать LLM-модели с внешними данными, инструментами и цепочками логики (chains), а также управлять контекстом, памятью и агентами для создания сложных, адаптивных приложений.


### Базовый уровень: messages → model → ответ

Это самый низкий уровень интерфейса, который всё ещё удобно использовать руками.

Сначала явно создаются `SystemMessage`, `HumanMessage`, затем они отправляются в `model.invoke(...)`.  
Так проще всего увидеть, что чат-модель на самом деле работает не со “строкой запроса”, а со списком сообщений.

Ещё один важный момент: объект ответа — это не просто текст.  
Обычно в нём есть метаданные, role, content и иногда дополнительные поля, которые потом используются агентами и графом.

In [6]:

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Ты пишешь очень короткие технические summaries."),
        ("human", "Суммаризируй тему: {topic}"),
    ]
)

print("ChatPromptTemplate class:")
print(ChatPromptTemplate)
print("\nPrompt object:")
print(prompt)

chain = prompt | model | StrOutputParser()

print("\nChain object:")
print(chain)

chain_result = chain.invoke({"topic": "tools, agents, memory и graph orchestration в LangChain"})
print("\nChain result:")
print(chain_result)


ChatPromptTemplate class:
<class 'langchain_core.prompts.chat.ChatPromptTemplate'>

Prompt object:
input_variables=['topic'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Ты пишешь очень короткие технические summaries.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Суммаризируй тему: {topic}'), additional_kwargs={})]

Chain object:
first=ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Ты пишешь очень короткие технические summaries.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Суммаризируй тему: {topic}'),

### LCEL-цепочка: prompt | model | parser

Здесь появляется главное удобство LangChain — композиция через оператор `|`.

Цепочка делает три вещи подряд:

1. заполняет шаблон `ChatPromptTemplate`;
2. отправляет результат в модель;
3. превращает ответ в обычную строку через `StrOutputParser`.

Это уже не “ручной вызов модели”, а переиспользуемый pipeline.  
Как только шагов становится больше одного, такой стиль обычно удобнее простых `invoke` над списками сообщений.

In [7]:

class ToolDecision(BaseModel):
    needs_tools: bool = Field(description="Нужны ли вообще tools для ответа")
    likely_tools: list[str] = Field(description="Какие tools, скорее всего, понадобятся")
    why: str = Field(description="Короткое объяснение")

print("ToolDecision class:")
print(ToolDecision)
print("\nToolDecision JSON schema:")
jprint(ToolDecision.model_json_schema())

decision_model = model.with_structured_output(ToolDecision)

decision = decision_model.invoke(
    "Пользователь спросил: сравни текущую погоду в Хельсинки и Таллине и посчитай разницу температур."
)

print("\nStructured output object type:", type(decision))
print("\nStructured output value:")
print(decision)


ToolDecision class:
<class '__main__.ToolDecision'>

ToolDecision JSON schema:
{
  "properties": {
    "needs_tools": {
      "description": "Нужны ли вообще tools для ответа",
      "title": "Needs Tools",
      "type": "boolean"
    },
    "likely_tools": {
      "description": "Какие tools, скорее всего, понадобятся",
      "items": {
        "type": "string"
      },
      "title": "Likely Tools",
      "type": "array"
    },
    "why": {
      "description": "Короткое объяснение",
      "title": "Why",
      "type": "string"
    }
  },
  "required": [
    "needs_tools",
    "likely_tools",
    "why"
  ],
  "title": "ToolDecision",
  "type": "object"
}

Structured output object type: <class '__main__.ToolDecision'>

Structured output value:
needs_tools=True likely_tools=['weather_api'] why='Чтобы сравнить текущую погоду в Хельсинки и Таллине и посчитать разницу температур, необходимо получить актуальные данные о температуре в обоих городах. Это требует использования погодного API, 

### Structured output до tools

На этом шаге модель ещё ничего не вызывает наружу, но уже обязана вернуть объект определённой формы.

Это очень полезный промежуточный уровень.  
До того как давать модели доступ к tools, можно сначала заставить её научиться возвращать валидную структуру: флаги, поля, списки, причины выбора.

В прикладных системах это часто один из лучших способов “сузить” поведение модели без жёсткого написания парсеров руками.

## Реальные HTTP API до LangChain tools

Перед тем как оборачивать что-то в `@tool`, полезно сначала написать обычные Python-функции и убедиться, что внешний API вообще работает как ожидается.

Это даёт сразу несколько преимуществ:

- можно отдельно отладить HTTP-запросы и формат ответа;
- можно понять, какие поля реально нужны агенту;
- можно не смешивать сетевую логику и агентную логику в одной функции.

Здесь сначала идут две сырьевые функции:

- `geocode_city_raw(...)` — находит координаты города через geocoding API;
- `get_weather_raw(...)` — получает текущую погоду и прогноз на день по координатам.

Такой слой почти всегда полезен и в production-коде: tools обычно становятся тонкой обёрткой поверх уже существующих функций доступа к данным.

In [8]:

def geocode_city_raw(city: str, country_code: str | None = None) -> dict[str, Any]:
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
    }
    if country_code:
        params["countryCode"] = country_code

    resp = SESSION.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    results = data.get("results") or []
    if not results:
        raise ValueError(f"Не удалось найти город: {city}")
    return results[0]


def get_weather_raw(
    latitude: float,
    longitude: float,
    timezone: str = "auto",
    forecast_days: int = 1,
) -> dict[str, Any]:
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "timezone": timezone,
        "forecast_days": forecast_days,
        "current": ",".join(
            [
                "temperature_2m",
                "apparent_temperature",
                "relative_humidity_2m",
                "wind_speed_10m",
                "weather_code",
            ]
        ),
        "daily": ",".join(
            [
                "temperature_2m_max",
                "temperature_2m_min",
                "precipitation_probability_max",
            ]
        ),
    }

    resp = SESSION.get(
        "https://api.open-meteo.com/v1/forecast",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def eval_mathjs_raw(expression: str) -> str:
    resp = SESSION.get(
        "https://api.mathjs.org/v4/",
        params={"expr": expression},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.text.strip()


In [9]:

helsinki_location = geocode_city_raw("Helsinki")
print("Geocoding result:")
jprint(
    {
        "name": helsinki_location["name"],
        "country": helsinki_location.get("country"),
        "latitude": helsinki_location["latitude"],
        "longitude": helsinki_location["longitude"],
        "timezone": helsinki_location.get("timezone"),
    }
)

helsinki_weather = get_weather_raw(
    latitude=helsinki_location["latitude"],
    longitude=helsinki_location["longitude"],
)

print("\nWeather result (trimmed):")
jprint(
    {
        "timezone": helsinki_weather["timezone"],
        "current": helsinki_weather["current"],
        "daily": {
            "time": helsinki_weather["daily"]["time"][0],
            "temperature_2m_max": helsinki_weather["daily"]["temperature_2m_max"][0],
            "temperature_2m_min": helsinki_weather["daily"]["temperature_2m_min"][0],
            "precipitation_probability_max": helsinki_weather["daily"]["precipitation_probability_max"][0],
        },
    }
)

print("\nmath.js result:")
print(eval_mathjs_raw("2^10 + sqrt(81)"))


Geocoding result:
{
  "name": "Helsingfors",
  "country": "Finland",
  "latitude": 60.16952,
  "longitude": 24.93545,
  "timezone": "Europe/Helsinki"
}

Weather result (trimmed):
{
  "timezone": "Europe/Helsinki",
  "current": {
    "time": "2026-03-18T12:00",
    "interval": 900,
    "temperature_2m": 1.5,
    "apparent_temperature": -3.7,
    "relative_humidity_2m": 96,
    "wind_speed_10m": 22.3,
    "weather_code": 3
  },
  "daily": {
    "time": "2026-03-18",
    "temperature_2m_max": 1.9,
    "temperature_2m_min": 0.4,
    "precipitation_probability_max": 0
  }
}

math.js result:
1033


### Что важно увидеть в сырых API-ответах

После этого блока уже понятно, какие поля нужны для дальнейшей работы:

- из geocoding нужны координаты и timezone;
- из weather API нужны текущие показатели и дневные min/max.

Хорошая практика — не прокидывать весь внешний JSON дальше без разбора.  
Обычно стоит сразу выделить минимальный полезный срез данных, чтобы tools и агент работали со стабильным форматом.

## Pydantic schemas + tools на реальных API

На этом шаге сырые функции превращаются в **инструменты**, с которыми уже может работать агент.

Ключевая идея: у tool должен быть **узкий и понятный контракт**.  
Именно поэтому сначала описывается вход через `BaseModel`, а уже потом функция оборачивается декоратором `@tool`.

Что здесь важно:

- `args_schema` даёт модели явную спецификацию аргументов;
- названия полей и `description` реально влияют на качество tool calling;
- tool лучше делать маленьким и однозначным, чем универсальным и размытым.

Отдельно полезно посмотреть на JSON Schema у каждого входного класса: именно примерно в таком виде сигнатура потом уходит в сторону модели.

In [10]:

class WeatherInput(BaseModel):
    city: str = Field(description="Название города, например Helsinki")
    country_code: str | None = Field(
        default=None,
        description="Необязательный ISO country code, например FI или EE",
    )


class CalculatorInput(BaseModel):
    expression: str = Field(
        description="Математическое выражение в синтаксисе math.js, например (17.2 - 12.8) * 1.8 + 32"
    )


print("WeatherInput class:")
print(WeatherInput)
print("\nWeatherInput schema:")
jprint(WeatherInput.model_json_schema())

print("\nCalculatorInput class:")
print(CalculatorInput)
print("\nCalculatorInput schema:")
jprint(CalculatorInput.model_json_schema())


WeatherInput class:
<class '__main__.WeatherInput'>

WeatherInput schema:
{
  "properties": {
    "city": {
      "description": "Название города, например Helsinki",
      "title": "City",
      "type": "string"
    },
    "country_code": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Необязательный ISO country code, например FI или EE",
      "title": "Country Code"
    }
  },
  "required": [
    "city"
  ],
  "title": "WeatherInput",
  "type": "object"
}

CalculatorInput class:
<class '__main__.CalculatorInput'>

CalculatorInput schema:
{
  "properties": {
    "expression": {
      "description": "Математическое выражение в синтаксисе math.js, например (17.2 - 12.8) * 1.8 + 32",
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "title": "CalculatorInput",
  "type": "object"
}


In [11]:

@tool(args_schema=WeatherInput)
def get_weather(city: str, country_code: str | None = None) -> str:
    """Get current weather and today's min/max forecast for a city using Open-Meteo."""
    location = geocode_city_raw(city, country_code=country_code)
    weather = get_weather_raw(
        latitude=location["latitude"],
        longitude=location["longitude"],
        timezone=location.get("timezone") or "auto",
        forecast_days=1,
    )

    result = {
        "city": location["name"],
        "country": location.get("country"),
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "timezone": weather["timezone"],
        "current": weather["current"],
        "today": {
            "date": weather["daily"]["time"][0],
            "temperature_2m_max": weather["daily"]["temperature_2m_max"][0],
            "temperature_2m_min": weather["daily"]["temperature_2m_min"][0],
            "precipitation_probability_max": weather["daily"]["precipitation_probability_max"][0],
        },
    }
    return json.dumps(result, ensure_ascii=False)


@tool(args_schema=CalculatorInput)
def evaluate_expression(expression: str) -> str:
    """Evaluate a math expression using the public math.js HTTP API."""
    value = eval_mathjs_raw(expression)
    return json.dumps({"expression": expression, "result": value}, ensure_ascii=False)


print("get_weather tool:")
print(get_weather)
print("\nget_weather tool type:", type(get_weather))
print("\nget_weather args schema:")
print(get_weather.args_schema)
print("\nget_weather args schema JSON:")
jprint(get_weather.args_schema.model_json_schema())

print("\nevaluate_expression tool:")
print(evaluate_expression)
print("\nevaluate_expression tool type:", type(evaluate_expression))
print("\nevaluate_expression args schema:")
print(evaluate_expression.args_schema)
print("\nevaluate_expression args schema JSON:")
jprint(evaluate_expression.args_schema.model_json_schema())


get_weather tool:
name='get_weather' description="Get current weather and today's min/max forecast for a city using Open-Meteo." args_schema=<class '__main__.WeatherInput'> func=<function get_weather at 0x7dec2b5393a0>

get_weather tool type: <class 'langchain_core.tools.structured.StructuredTool'>

get_weather args schema:
<class '__main__.WeatherInput'>

get_weather args schema JSON:
{
  "properties": {
    "city": {
      "description": "Название города, например Helsinki",
      "title": "City",
      "type": "string"
    },
    "country_code": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Необязательный ISO country code, например FI или EE",
      "title": "Country Code"
    }
  },
  "required": [
    "city"
  ],
  "title": "WeatherInput",
  "type": "object"
}

evaluate_expression tool:
name='evaluate_expression' description='Evaluate a math expression using the 

In [12]:

print("Direct tool call: get_weather")
print(get_weather.invoke({"city": "Tallinn", "country_code": "EE"}))

print("\nDirect tool call: evaluate_expression")
print(evaluate_expression.invoke({"expression": "(17.2 - 12.8) * 1.8 + 32"}))


Direct tool call: get_weather
{"city": "Revel", "country": "Estonia", "latitude": 59.43696, "longitude": 24.75353, "timezone": "Europe/Tallinn", "current": {"time": "2026-03-18T12:00", "interval": 900, "temperature_2m": 2.7, "apparent_temperature": -1.5, "relative_humidity_2m": 85, "wind_speed_10m": 14.4, "weather_code": 3}, "today": {"date": "2026-03-18", "temperature_2m_max": 6.1, "temperature_2m_min": 0.6, "precipitation_probability_max": 0}}

Direct tool call: evaluate_expression
{"expression": "(17.2 - 12.8) * 1.8 + 32", "result": "39.919999999999995"}


### Прямой вызов tools без агента

Это важный диагностический этап.  
Если tool корректно вызывается напрямую через `.invoke(...)`, значит схема аргументов, обёртка и внутренняя функция уже согласованы между собой.

Так удобно проверять:

- проходит ли валидация входа;
- что именно возвращает tool;
- не потерялись ли важные поля;
- удобно ли формулировать docstring и descriptions.

Отладка tools почти всегда должна начинаться именно отсюда, а не с полноценного агентного прогона.

## `create_agent` как high-level оболочка и почему здесь нужен compat mode

`create_agent(...)` всё равно важно посмотреть, потому что это нормальная high-level точка входа в LangChain для tool-using агента.  
Даже если текущий backend не поддерживает server-side auto tool choice, сам объект агента полезно создать и проинспектировать: видно, что LangChain собирает граф исполнения поверх LangGraph.

Но исполнять tool loop через `agent.invoke(...)` на таком backend нельзя.  
Поэтому ниже делается честный compat mode:

- агент создаётся и его граф печатается;
- сами tools вызываются вручную;
- планирование и финальная сборка ответа делаются через модель и structured output.

С практической точки зрения это не обход “для галочки”, а нормальный production-паттерн.  
Когда tool-calling на стороне сервера ограничен, полезно явно разделить planning, execution и state.

In [13]:
agent = create_agent(
    model=model,
    tools=[get_weather, evaluate_expression],
    system_prompt=(
        "Ты инженерный агент. "
        "Для погоды всегда используй get_weather. "
        "Для арифметики всегда используй evaluate_expression. "
        "Если надо сравнить температуры, сначала получи данные tool-вызовами, "
        "потом передай чистое выражение в evaluate_expression."
    ),
)

print("Agent object type:")
print(type(agent))

print("\nAgent graph (Mermaid):")
print(agent.get_graph().draw_mermaid())

print("\nНиже tools будут вызываться вручную, потому что текущий backend не поддерживает auto tool choice.")

Agent object type:
<class 'langgraph.graph.state.CompiledStateGraph'>

Agent graph (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


Ниже tools будут вызываться вручную, потому что текущий backend не поддерживает auto tool choice.


In [14]:
def extract_weather_payload(tool_output: str) -> dict[str, Any]:
    payload = json.loads(tool_output)
    print("Parsed weather payload keys:", payload.keys())
    return payload


def extract_current_temperature_c(tool_output: str) -> float:
    payload = extract_weather_payload(tool_output)
    temperature = float(payload["current"]["temperature_2m"])
    print("Current temperature:", temperature)
    return temperature


def compare_weather_manual(
    city_a: str,
    city_b: str,
    country_code_a: str | None = None,
    country_code_b: str | None = None,
) -> dict[str, Any]:
    weather_a_raw = get_weather.invoke({"city": city_a, "country_code": country_code_a})
    weather_b_raw = get_weather.invoke({"city": city_b, "country_code": country_code_b})

    print("Raw tool output A:")
    print(weather_a_raw)
    print("\nRaw tool output B:")
    print(weather_b_raw)

    weather_a = json.loads(weather_a_raw)
    weather_b = json.loads(weather_b_raw)

    temp_a = float(weather_a["current"]["temperature_2m"])
    temp_b = float(weather_b["current"]["temperature_2m"])

    diff_raw = evaluate_expression.invoke(
        {"expression": f"abs({temp_a} - {temp_b})"}
    )
    diff_payload = json.loads(diff_raw)
    diff_c = float(diff_payload["result"])

    warmer_city = weather_a["city"] if temp_a >= temp_b else weather_b["city"]
    colder_city = weather_b["city"] if temp_a >= temp_b else weather_a["city"]

    final_message = model.invoke(
        [
            SystemMessage(
                content=(
                    "Собери короткий инженерный ответ по-русски. "
                    "Скажи, где теплее, укажи температуры и разницу."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "city_a_weather": weather_a,
                        "city_b_weather": weather_b,
                        "difference_payload": diff_payload,
                        "warmer_city": warmer_city,
                        "colder_city": colder_city,
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )

    return {
        "weather_a": weather_a,
        "weather_b": weather_b,
        "temp_a": temp_a,
        "temp_b": temp_b,
        "difference_payload": diff_payload,
        "difference_c": diff_c,
        "warmer_city": warmer_city,
        "colder_city": colder_city,
        "final_message": final_message,
    }


manual_compare_result = compare_weather_manual(
    "Helsinki",
    "Tallinn",
    country_code_a="FI",
    country_code_b="EE",
)

print("\nSummary fields:")
jprint(
    {
        "warmer_city": manual_compare_result["warmer_city"],
        "colder_city": manual_compare_result["colder_city"],
        "difference_c": manual_compare_result["difference_c"],
    }
)

print("\nFinal model answer:")
show_message(manual_compare_result["final_message"])

Raw tool output A:
{"city": "Helsingfors", "country": "Finland", "latitude": 60.16952, "longitude": 24.93545, "timezone": "Europe/Helsinki", "current": {"time": "2026-03-18T12:00", "interval": 900, "temperature_2m": 1.5, "apparent_temperature": -3.7, "relative_humidity_2m": 96, "wind_speed_10m": 22.3, "weather_code": 3}, "today": {"date": "2026-03-18", "temperature_2m_max": 1.9, "temperature_2m_min": 0.4, "precipitation_probability_max": 0}}

Raw tool output B:
{"city": "Revel", "country": "Estonia", "latitude": 59.43696, "longitude": 24.75353, "timezone": "Europe/Tallinn", "current": {"time": "2026-03-18T12:00", "interval": 900, "temperature_2m": 2.7, "apparent_temperature": -1.5, "relative_humidity_2m": 85, "wind_speed_10m": 14.4, "weather_code": 3}, "today": {"date": "2026-03-18", "temperature_2m_max": 6.1, "temperature_2m_min": 0.6, "precipitation_probability_max": 0}}

Summary fields:
{
  "warmer_city": "Revel",
  "colder_city": "Helsingfors",
  "difference_c": 1.2000000000000002


In [15]:
class ManualToolPlan(BaseModel):
    weather_cities: list[str] = Field(
        default_factory=list,
        description="Список городов, для которых нужно получить погоду",
    )
    expression: str | None = Field(
        default=None,
        description="Математическое выражение для evaluate_expression",
    )
    final_task: str = Field(
        description="Что именно надо сделать в финальном ответе после tool-вызовов",
    )


print("ManualToolPlan class:")
print(ManualToolPlan)
print("\nManualToolPlan schema:")
jprint(ManualToolPlan.model_json_schema())

manual_planner = model.with_structured_output(ManualToolPlan)


def manual_tool_stream(user_query: str):
    plan = manual_planner.invoke(
        [
            SystemMessage(
                content=(
                    "Составь план tool execution. "
                    "Верни weather_cities, expression и final_task. "
                    "Если погода не нужна, верни пустой список. "
                    "Если математика не нужна, верни null."
                )
            ),
            HumanMessage(content=user_query),
        ]
    )
    yield {"planner": plan.model_dump()}

    weather_results: dict[str, Any] = {}
    for city in plan.weather_cities:
        tool_output = get_weather.invoke({"city": city})
        payload = json.loads(tool_output)
        weather_results[city] = payload
        yield {
            "tool": {
                "name": "get_weather",
                "city": city,
                "temperature_2m": payload["current"]["temperature_2m"],
                "timezone": payload["timezone"],
            }
        }

    calc_payload = None
    if plan.expression:
        calc_output = evaluate_expression.invoke({"expression": plan.expression})
        calc_payload = json.loads(calc_output)
        yield {"tool": {"name": "evaluate_expression", "payload": calc_payload}}

    final = model.invoke(
        [
            SystemMessage(
                content=(
                    "Собери короткий технический итог. "
                    "Используй только полученные tool results."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "user_query": user_query,
                        "plan": plan.model_dump(),
                        "weather_results": weather_results,
                        "calc_payload": calc_payload,
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    yield {"final_message": final.content}


print("Streaming manual updates:")
for step in manual_tool_stream(
    "Какая погода сейчас в Berlin и сколько будет 3 * (7 + 5)?"
):
    jprint(step)

ManualToolPlan class:
<class '__main__.ManualToolPlan'>

ManualToolPlan schema:
{
  "properties": {
    "weather_cities": {
      "description": "Список городов, для которых нужно получить погоду",
      "items": {
        "type": "string"
      },
      "title": "Weather Cities",
      "type": "array"
    },
    "expression": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Математическое выражение для evaluate_expression",
      "title": "Expression"
    },
    "final_task": {
      "description": "Что именно надо сделать в финальном ответе после tool-вызовов",
      "title": "Final Task",
      "type": "string"
    }
  },
  "required": [
    "final_task"
  ],
  "title": "ManualToolPlan",
  "type": "object"
}
Streaming manual updates:
{
  "planner": {
    "weather_cities": [
      "Berlin"
    ],
    "expression": "3 * (7 + 5)",
    "final_task": "Определить текущую по

### Инспекция high-level агента и ручной compat workflow

Граф агента всё равно полезно печатать.  
Так видно, как LangChain концептуально организует execution loop: есть модельный шаг, есть tools, есть переходы между ними.

Но на текущем backend безопаснее и честнее запускать не `agent.stream(...)`, а собственный пошаговый runner.  
Так остаются все важные части архитектуры:

- structured planning;
- реальные tool-вызовы;
- промежуточные шаги;
- финальная сборка ответа моделью.

По сути это тот же agent-like workflow, только без зависимости от server-side auto tool choice.

## Structured output после ручных tool-вызовов

Следующий полезный слой — не просто текстовый ответ, а объект фиксированной формы.  
На практике это часто оказывается надёжнее, чем пытаться одновременно получать и tool execution, и строгий JSON из одного агентного вызова.

Паттерн здесь такой:

1. tools возвращают реальные данные;
2. если нужна арифметика, её делает отдельный calculator tool;
3. затем модель получает уже готовые факты и возвращает строго типизированный объект.

Так проще валидировать данные, логировать их и использовать дальше в коде.

In [16]:
class WeatherComparison(BaseModel):
    warmer_city: str = Field(description="Город, где сейчас теплее")
    colder_city: str = Field(description="Город, где сейчас холоднее")
    temperature_difference_c: float = Field(description="Разница температур в Цельсиях")
    summary: str = Field(description="Короткая текстовая сводка")

print("WeatherComparison class:")
print(WeatherComparison)
print("\nWeatherComparison schema:")
jprint(WeatherComparison.model_json_schema())

comparison_model = model.with_structured_output(WeatherComparison)

riga_weather_raw = get_weather.invoke({"city": "Riga", "country_code": "LV"})
vilnius_weather_raw = get_weather.invoke({"city": "Vilnius", "country_code": "LT"})

riga_weather = json.loads(riga_weather_raw)
vilnius_weather = json.loads(vilnius_weather_raw)

riga_temp = float(riga_weather["current"]["temperature_2m"])
vilnius_temp = float(vilnius_weather["current"]["temperature_2m"])

difference_raw = evaluate_expression.invoke(
    {"expression": f"abs({riga_temp} - {vilnius_temp})"}
)
difference_payload = json.loads(difference_raw)

structured_comparison = comparison_model.invoke(
    [
        SystemMessage(
            content=(
                "Верни объект WeatherComparison. "
                "Используй только переданные факты из tools."
            )
        ),
        HumanMessage(
            content=json.dumps(
                {
                    "riga_weather": riga_weather,
                    "vilnius_weather": vilnius_weather,
                    "difference_payload": difference_payload,
                },
                ensure_ascii=False,
            )
        ),
    ]
)

print("Structured comparison object type:", type(structured_comparison))
print("\nStructured comparison:")
print(structured_comparison)

WeatherComparison class:
<class '__main__.WeatherComparison'>

WeatherComparison schema:
{
  "properties": {
    "warmer_city": {
      "description": "Город, где сейчас теплее",
      "title": "Warmer City",
      "type": "string"
    },
    "colder_city": {
      "description": "Город, где сейчас холоднее",
      "title": "Colder City",
      "type": "string"
    },
    "temperature_difference_c": {
      "description": "Разница температур в Цельсиях",
      "title": "Temperature Difference C",
      "type": "number"
    },
    "summary": {
      "description": "Короткая текстовая сводка",
      "title": "Summary",
      "type": "string"
    }
  },
  "required": [
    "warmer_city",
    "colder_city",
    "temperature_difference_c",
    "summary"
  ],
  "title": "WeatherComparison",
  "type": "object"
}
Structured comparison object type: <class '__main__.WeatherComparison'>

Structured comparison:
warmer_city='Vilnius' colder_city='Riga' temperature_difference_c=3.7 summary='Vilnius 

In [17]:

print("AgentState class:")
print(AgentState)
print("\nAgentState annotations:")
print(AgentState.__annotations__)


class CustomAgentState(AgentState):
    home_city: str | None
    preferred_unit: str | None


print("\nCustomAgentState class:")
print(CustomAgentState)
print("\nCustomAgentState annotations:")
print(CustomAgentState.__annotations__)


AgentState class:
<class 'langchain.agents.middleware.types.AgentState'>

AgentState annotations:
{'messages': ForwardRef('Required[Annotated[list[AnyMessage], add_messages]]', module='langchain.agents.middleware.types'), 'jump_to': ForwardRef('NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]', module='langchain.agents.middleware.types'), 'structured_response': ForwardRef('NotRequired[Annotated[ResponseT, OmitFromInput]]', module='langchain.agents.middleware.types')}

CustomAgentState class:
<class '__main__.CustomAgentState'>

CustomAgentState annotations:
{'messages': ForwardRef('Required[Annotated[list[AnyMessage], add_messages]]', module='langchain.agents.middleware.types'), 'jump_to': ForwardRef('NotRequired[Annotated[JumpTo | None, EphemeralValue, PrivateStateAttr]]', module='langchain.agents.middleware.types'), 'structured_response': ForwardRef('NotRequired[Annotated[ResponseT, OmitFromInput]]', module='langchain.agents.middleware.types'), 'home_city': For

### Зачем смотреть на `AgentState`

Когда появляется память, быстро выясняется, что агент — это уже не просто функция из текста в текст.  
У него есть состояние: сообщения, промежуточные значения, служебные поля и иногда собственные пользовательские данные.

Поэтому здесь полезно явно посмотреть на `AgentState` и затем расширить его собственным классом.  
Даже если в этой версии блок не использует кастомный state в полном объёме, сам паттерн важен: состояние можно типизировать, расширять и делать более прозрачным.

## Short-term memory через `InMemorySaver` + `thread_id` без tool-агента

Память в LangGraph не привязана только к агентам.  
На самом деле `checkpointer` работает на уровне графа состояния, поэтому показать память удобнее на простом графе с одним chat-node.

Это полезный сдвиг в мышлении.  
Память — не “способность модели помнить”, а сохранение state между вызовами одного и того же графа.

В этом блоке используется самый простой вариант:

- `MessagesState` как состояние;
- один node, который вызывает модель;
- `InMemorySaver` как checkpointer;
- один и тот же `thread_id` для двух последовательных вызовов.

In [18]:
print("InMemorySaver class:")
print(InMemorySaver)
print("\nInMemorySaver signature:")
print(inspect.signature(InMemorySaver))

checkpointer = InMemorySaver()


def chat_node(state: MessagesState) -> dict[str, Any]:
    response = model.invoke(
        [
            SystemMessage(
                content=(
                    "Отвечай кратко и опирайся на всю историю сообщений в текущем thread."
                )
            ),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


memory_builder = StateGraph(MessagesState)
memory_builder.add_node("chat", chat_node)
memory_builder.add_edge(START, "chat")
memory_builder.add_edge("chat", END)

memory_graph = memory_builder.compile(checkpointer=checkpointer)

print("\nMemory graph type:")
print(type(memory_graph))
print("\nMemory graph (Mermaid):")
print(memory_graph.get_graph().draw_mermaid())

thread = {"configurable": {"thread_id": "seminar-6-memory-demo"}}

InMemorySaver class:
<class 'langgraph.checkpoint.memory.InMemorySaver'>

InMemorySaver signature:
(*, serde: 'SerializerProtocol | None' = None, factory: 'type[defaultdict]' = <class 'collections.defaultdict'>) -> 'None'

Memory graph type:
<class 'langgraph.graph.state.CompiledStateGraph'>

Memory graph (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chat(chat)
	__end__([<p>__end__</p>]):::last
	__start__ --> chat;
	chat --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [19]:
memory_graph.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Запомни, что мне удобнее ответы в Цельсиях, "
                    "и мой домашний город — Helsinki."
                )
            )
        ]
    },
    config=thread,
)

memory_result = memory_graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="Какой у меня домашний город и в каких единицах мне отвечать?"
            )
        ]
    },
    config=thread,
)

show_last_message(memory_result)

================================== Ai Message ==================================

Твой домашний город — Хельсинки. Отвечаю в градусах Цельсия.


In [20]:
snapshot = memory_graph.get_state(thread)

print("State snapshot type:", type(snapshot))
print("\nState snapshot fields:")
print(snapshot.__annotations__.keys())

print("\nState keys inside snapshot.values:")
print(snapshot.values.keys())

stored_messages = snapshot.values.get("messages", [])

print("\nHow many messages are stored in this thread?")
print(len(stored_messages))

print("\nLast 4 messages from stored thread:")
for msg in stored_messages[-4:]:
    show_message(msg)
    print("-" * 80)

State snapshot type: <class 'langgraph.types.StateSnapshot'>

State snapshot fields:
dict_keys(['values', 'next', 'config', 'metadata', 'created_at', 'parent_config', 'tasks', 'interrupts'])

State keys inside snapshot.values:
dict_keys(['messages'])

How many messages are stored in this thread?
4

Last 4 messages from stored thread:
================================ Human Message =================================

Запомни, что мне удобнее ответы в Цельсиях, и мой домашний город — Helsinki.
--------------------------------------------------------------------------------
================================== Ai Message ==================================

Запомнил: предпочитаешь температуры в Цельсиях, и твой домашний город — Хельсинки.
--------------------------------------------------------------------------------
================================ Human Message =================================

Какой у меня домашний город и в каких единицах мне отвечать?
-----------------------------------

### Что здесь важно увидеть в памяти

Здесь память работает уже без tool-агента и без специальных “memory prompt” трюков.  
Граф просто поднимает прошлое состояние по `thread_id`, добавляет новые сообщения и снова прогоняет node.

Это удобно по двум причинам:

- становится понятно, что short-term memory — это именно persisted graph state;
- тот же подход потом можно применять и к более сложным workflow, а не только к чату.

## Кастомный workflow в `StateGraph`

Агент удобен, когда логика может оставаться довольно свободной.  
Но как только нужен **явный порядок шагов**, лучше собрать граф самостоятельно.

`StateGraph` полезен, когда хочется контролировать:

- какие узлы вообще существуют;
- в каком порядке они запускаются;
- какие поля состояния появляются после каждого шага;
- где логика должна быть детерминированной, а не агентной.

В этом блоке граф собирается вручную: сначала планирование, потом weather step, потом calculator step, потом финальная сборка ответа.  
Это хороший пример того, где LangGraph даёт больше прозрачности, чем “чёрный ящик агента”.

In [21]:

print("MessagesState class:")
print(MessagesState)
print("\nMessagesState annotations:")
print(MessagesState.__annotations__)

print("\nStateGraph class:")
print(StateGraph)
print("\nStateGraph signature:")
print(inspect.signature(StateGraph))


MessagesState class:
<class 'langgraph.graph.message.MessagesState'>

MessagesState annotations:
{'messages': ForwardRef('Annotated[list[AnyMessage], add_messages]', module='langgraph.graph.message')}

StateGraph class:
<class 'langgraph.graph.state.StateGraph'>

StateGraph signature:
(state_schema: 'type[StateT]', context_schema: 'type[ContextT] | None' = None, *, input_schema: 'type[InputT] | None' = None, output_schema: 'type[OutputT] | None' = None, **kwargs: 'Unpack[DeprecatedKwargs]') -> 'None'


In [22]:

class WorkflowPlan(BaseModel):
    city: str | None = Field(default=None, description="Город для weather step")
    expression: str | None = Field(default=None, description="Математическое выражение для calculator step")

print("WorkflowPlan class:")
print(WorkflowPlan)
print("\nWorkflowPlan schema:")
jprint(WorkflowPlan.model_json_schema())


class WeatherWorkflowState(MessagesState):
    city: str | None
    weather_json: str | None
    expression: str | None
    calc_json: str | None

print("\nWeatherWorkflowState class:")
print(WeatherWorkflowState)
print("\nWeatherWorkflowState annotations:")
print(WeatherWorkflowState.__annotations__)

planner_model = model.with_structured_output(WorkflowPlan)


WorkflowPlan class:
<class '__main__.WorkflowPlan'>

WorkflowPlan schema:
{
  "properties": {
    "city": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Город для weather step",
      "title": "City"
    },
    "expression": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Математическое выражение для calculator step",
      "title": "Expression"
    }
  },
  "title": "WorkflowPlan",
  "type": "object"
}

WeatherWorkflowState class:
<class '__main__.WeatherWorkflowState'>

WeatherWorkflowState annotations:
{'messages': ForwardRef('Annotated[list[AnyMessage], add_messages]', module='langgraph.graph.message'), 'city': ForwardRef('str | None', module='__main__'), 'weather_json': ForwardRef('str | None', module='__main__'), 'expression': ForwardRef('str | No

In [23]:

def plan_node(state: WeatherWorkflowState) -> dict[str, Any]:
    user_text = state["messages"][-1].content
    plan = planner_model.invoke(
        [
            SystemMessage(
                content=(
                    "Извлеки из запроса два поля: "
                    "city для weather step и expression для calculator step. "
                    "Если чего-то нет, верни null."
                )
            ),
            HumanMessage(content=str(user_text)),
        ]
    )
    return {
        "city": plan.city,
        "expression": plan.expression,
    }


def weather_node(state: WeatherWorkflowState) -> dict[str, Any]:
    if not state.get("city"):
        return {}
    return {
        "weather_json": get_weather.invoke({"city": state["city"]}),
    }


def calculator_node(state: WeatherWorkflowState) -> dict[str, Any]:
    if not state.get("expression"):
        return {}
    return {
        "calc_json": evaluate_expression.invoke({"expression": state["expression"]}),
    }


def final_node(state: WeatherWorkflowState) -> dict[str, Any]:
    final_answer = model.invoke(
        [
            SystemMessage(
                content=(
                    "Собери короткий итоговый ответ. "
                    "Если есть weather_json, используй его. "
                    "Если есть calc_json, используй его."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "user_request": state["messages"][-1].content,
                        "city": state.get("city"),
                        "weather_json": state.get("weather_json"),
                        "expression": state.get("expression"),
                        "calc_json": state.get("calc_json"),
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    return {"messages": [final_answer]}


In [24]:

workflow_builder = StateGraph(WeatherWorkflowState)
workflow_builder.add_node("plan", plan_node)
workflow_builder.add_node("weather", weather_node)
workflow_builder.add_node("calculator", calculator_node)
workflow_builder.add_node("final", final_node)

workflow_builder.add_edge(START, "plan")
workflow_builder.add_edge("plan", "weather")
workflow_builder.add_edge("weather", "calculator")
workflow_builder.add_edge("calculator", "final")
workflow_builder.add_edge("final", END)

workflow = workflow_builder.compile(checkpointer=InMemorySaver())

print(workflow.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	weather(weather)
	calculator(calculator)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	calculator --> final;
	plan --> weather;
	weather --> calculator;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [25]:

workflow_result = workflow.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Для города Prague возьми текущую погоду, "
                    "а отдельно вычисли выражение 14^2 - 17."
                )
            )
        ]
    },
    config={"configurable": {"thread_id": "workflow-demo-1"}},
)

show_last_message(workflow_result)


================================== Ai Message ==================================

В Праге (Чехия) сейчас 8.3°C, ощущается как 5.3°C, влажность 56%, скорость ветра 6.6 м/с, погода ясная (код 0). Максимальная температура сегодня — 14.9°C, минимальная — 1.3°C.  
Результат выражения 14² - 17 = 179.


### Почему этот блок уже больше похож на workflow engine

После запуска графа становится видно отличие от агента.

Здесь нет свободного выбора следующего шага на каждом витке.  
Есть заранее определённая структура: планирование → weather → calculator → финальная сборка.  
То есть модель используется как часть workflow, а не как единственный источник логики.

Это важный production-переход: модель может отвечать за интерпретацию, но архитектурный контроль остаётся у графа.

## Long-term memory через `InMemoryStore` и отдельный profile workflow

Теперь нужен другой тип памяти.  
История треда отвечает только за текущий поток сообщений, а профиль пользователя должен жить дольше конкретного разговора.

Поэтому ниже делается отдельный workflow для long-term memory:

- `store` хранит preferences по namespace;
- модель сначала решает, нужно сохранить preference или прочитать уже сохранённые;
- дальше graph вызывает обычные Python-функции работы со store;
- финальный ответ снова собирается моделью.

Это уже хороший production-паттерн: долговременные данные лежат отдельно от chat history.

In [26]:
print("InMemoryStore class:")
print(InMemoryStore)
print("\nInMemoryStore signature:")
print(inspect.signature(InMemoryStore))

store = InMemoryStore()

store.put(
    ("users", "andrey", "preferences"),
    "temperature_unit",
    {"value": "celsius"},
)
store.put(
    ("users", "andrey", "preferences"),
    "home_city",
    {"value": "Helsinki"},
)

items = store.search(("users", "andrey", "preferences"))

print("\nDirect store search:")
for item in items:
    print(
        {
            "namespace": item.namespace,
            "key": item.key,
            "value": item.value,
        }
    )

InMemoryStore class:
<class 'langgraph.store.memory.InMemoryStore'>

InMemoryStore signature:
(*, index: 'IndexConfig | None' = None) -> 'None'

Direct store search:
{'namespace': ('users', 'andrey', 'preferences'), 'key': 'temperature_unit', 'value': {'value': 'celsius'}}
{'namespace': ('users', 'andrey', 'preferences'), 'key': 'home_city', 'value': {'value': 'Helsinki'}}


In [27]:
@dataclass
class UserContext:
    user_id: str


class PreferenceAction(BaseModel):
    action: str = Field(
        description="save если нужно сохранить preference, read если нужно прочитать preferences"
    )
    key: str | None = Field(default=None, description="Ключ preference")
    value: str | None = Field(default=None, description="Значение preference")


print("UserContext class:")
print(UserContext)

print("\nPreferenceAction class:")
print(PreferenceAction)
print("\nPreferenceAction schema:")
jprint(PreferenceAction.model_json_schema())


class ProfileState(MessagesState):
    user_id: str
    action: str | None
    key: str | None
    value: str | None
    preferences_json: str | None


print("\nProfileState class:")
print(ProfileState)
print("\nProfileState annotations:")
print(ProfileState.__annotations__)


def preference_namespace(user_id: str) -> tuple[str, str, str]:
    return ("users", user_id, "preferences")


def save_preference_to_store(user_id: str, key: str, value: str) -> dict[str, Any]:
    namespace = preference_namespace(user_id)
    store.put(namespace, key, {"value": value})
    return {
        "status": "saved",
        "user_id": user_id,
        "key": key,
        "value": value,
    }


def read_preferences_from_store(user_id: str) -> dict[str, Any]:
    namespace = preference_namespace(user_id)
    items = store.search(namespace)
    return {item.key: item.value["value"] for item in items}


profile_planner = model.with_structured_output(PreferenceAction)

UserContext class:
<class '__main__.UserContext'>

PreferenceAction class:
<class '__main__.PreferenceAction'>

PreferenceAction schema:
{
  "properties": {
    "action": {
      "description": "save если нужно сохранить preference, read если нужно прочитать preferences",
      "title": "Action",
      "type": "string"
    },
    "key": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Ключ preference",
      "title": "Key"
    },
    "value": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Значение preference",
      "title": "Value"
    }
  },
  "required": [
    "action"
  ],
  "title": "PreferenceAction",
  "type": "object"
}

ProfileState class:
<class '__main__.ProfileState'>

ProfileState annotations:
{'messages': ForwardRef('Annotated[list[AnyMessa

In [28]:
def profile_route_node(state: ProfileState) -> dict[str, Any]:
    user_text = state["messages"][-1].content
    plan = profile_planner.invoke(
        [
            SystemMessage(
                content=(
                    "Определи действие с пользовательскими preferences. "
                    "Верни action=save если нужно что-то запомнить. "
                    "Верни action=read если нужно прочитать preferences. "
                    "Если action=save, постарайся извлечь один key и один value."
                )
            ),
            HumanMessage(content=str(user_text)),
        ]
    )
    return {
        "action": plan.action,
        "key": plan.key,
        "value": plan.value,
    }


def profile_store_node(state: ProfileState) -> dict[str, Any]:
    user_id = state["user_id"]

    if state.get("action") == "save" and state.get("key") and state.get("value"):
        payload = save_preference_to_store(
            user_id=user_id,
            key=state["key"],
            value=state["value"],
        )
    else:
        payload = read_preferences_from_store(user_id=user_id)

    return {"preferences_json": json.dumps(payload, ensure_ascii=False)}


def profile_final_node(state: ProfileState) -> dict[str, Any]:
    final_answer = model.invoke(
        [
            SystemMessage(
                content=(
                    "Собери короткий ответ по-русски. "
                    "Если preferences только что сохранены, явно скажи это. "
                    "Если preferences прочитаны, перечисли их."
                )
            ),
            HumanMessage(
                content=json.dumps(
                    {
                        "user_id": state["user_id"],
                        "action": state.get("action"),
                        "key": state.get("key"),
                        "value": state.get("value"),
                        "preferences_json": state.get("preferences_json"),
                        "user_request": state["messages"][-1].content,
                    },
                    ensure_ascii=False,
                )
            ),
        ]
    )
    return {"messages": [final_answer]}


profile_builder = StateGraph(ProfileState)
profile_builder.add_node("route", profile_route_node)
profile_builder.add_node("store_step", profile_store_node)
profile_builder.add_node("final", profile_final_node)

profile_builder.add_edge(START, "route")
profile_builder.add_edge("route", "store_step")
profile_builder.add_edge("store_step", "final")
profile_builder.add_edge("final", END)

profile_graph = profile_builder.compile()

print("Profile graph type:")
print(type(profile_graph))
print("\nProfile graph (Mermaid):")
print(profile_graph.get_graph().draw_mermaid())

Profile graph type:
<class 'langgraph.graph.state.CompiledStateGraph'>

Profile graph (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	store_step(store_step)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	route --> store_step;
	store_step --> final;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [29]:
profile_graph.invoke(
    {
        "user_id": "andrey",
        "messages": [
            HumanMessage(
                content=(
                    "Запомни, что мне нужны ответы кратко "
                    "и мой любимый город для weather demo — Tallinn."
                )
            )
        ],
    }
)

andrey_result = profile_graph.invoke(
    {
        "user_id": "andrey",
        "messages": [
            HumanMessage(content="Какие мои сохранённые preferences?")
        ],
    }
)

guest_result = profile_graph.invoke(
    {
        "user_id": "guest",
        "messages": [
            HumanMessage(content="Какие мои сохранённые preferences?")
        ],
    }
)

print("Andrey:")
show_last_message(andrey_result)

print("\nGuest:")
show_last_message(guest_result)

print("\nRaw store items for andrey:")
for item in store.search(("users", "andrey", "preferences")):
    print({"key": item.key, "value": item.value})

Andrey:
================================== Ai Message ==================================

Прочитаны следующие preferences:  
- temperature_unit: celsius  
- home_city: Helsinki  
- preferred_city: Tallinn

Guest:
================================== Ai Message ==================================

Ваши сохранённые preferences: пусто ({}).

Raw store items for andrey:
{'key': 'temperature_unit', 'value': {'value': 'celsius'}}
{'key': 'home_city', 'value': {'value': 'Helsinki'}}
{'key': 'preferred_city', 'value': {'value': 'Tallinn'}}


### Почему long-term memory здесь вынесена в отдельный workflow

Такой вариант проще контролировать, чем пытаться смешать всё в один универсальный agent loop.

Здесь каждая часть прозрачна:

- planner решает save или read;
- store-step делает реальную запись или чтение;
- final-node превращает результат в нормальный ответ.

Если позже понадобится база, Redis или другой persistence layer, менять придётся именно store-операции, а не всю логику чата.

## Минимальный cheat sheet

Ниже — вся лестница ещё раз в сжатом виде.  
Это уже версия с поправкой на backend, где auto tool choice недоступен, поэтому orchestration местами показана вручную.

- model → `ChatOpenAI(...)`
- prompt → `ChatPromptTemplate`
- plain chain → `prompt | model | StrOutputParser()`
- typed output → `model.with_structured_output(...)`
- tool → `@tool`
- high-level agent shape → `create_agent(...)`
- compat orchestration → planner + direct tool calls + final model step
- short-term memory → `checkpointer=InMemorySaver()` + `thread_id`
- custom orchestration → `StateGraph(...)`
- long-term memory → `store=InMemoryStore()`

Практическое правило здесь такое:

- если нужен один аккуратный вызов — хватает chain;
- если backend умеет normal tool calling — можно поднимать полноценный agent loop;
- если backend ограничен — planning и tools лучше разделить вручную;
- если нужен явный порядок шагов — нужен `StateGraph`;
- если нужно помнить пользователя между сессиями — нужен `store`.